In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Data Quality & Initial Exploration

Before performing SQL analysis and building the Power BI dashboard, the dataset is inspected for missing values, duplicates, cancelled transactions, invalid quantities and prices, and other potential data-quality issues.

The objective is to understand the structure and quality of the transactional data before cleaning and loading it into the analytical workflow.

In [2]:
df=pd.read_excel(r"C:\Users\tg_mu\Data Science\ChatGPT_Data Scientist\E-Commerce Sales & Customer Analytics\Online Retail.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
df.shape

(541909, 8)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [5]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(5268)

In [7]:
# Check cancelled invoices

cancelled = df['InvoiceNo'].astype(str).str.startswith('C')

print("Cancelled transactions:", cancelled.sum())
print(f"Percentage cancelled: {cancelled.mean() * 100:.2f}%")

Cancelled transactions: 9288
Percentage cancelled: 1.71%


In [8]:
# Check negative quantities

negative_quantity = (df['Quantity'] < 0).sum()

print("Negative quantity transactions:", negative_quantity)

Negative quantity transactions: 10624


In [9]:
# Check zero or negative unit prices

invalid_price = (df['UnitPrice'] <= 0).sum()

print("Zero/negative price transactions:", invalid_price)

Zero/negative price transactions: 2517


In [10]:
# Relationship between cancelled invoices and negative quantities

cancelled = df['InvoiceNo'].astype(str).str.startswith('C')
negative_qty = df['Quantity'] < 0

print("Cancelled invoices:", cancelled.sum())
print("Negative quantity:", negative_qty.sum())
print("Both cancelled and negative quantity:", (cancelled & negative_qty).sum())

Cancelled invoices: 9288
Negative quantity: 10624
Both cancelled and negative quantity: 9288


# 3. Data Cleaning

The transactional dataset contains duplicate records, cancelled invoices, negative quantities, and invalid unit prices.

For the primary sales analysis, cancelled transactions, non-positive quantities, and non-positive unit prices are excluded. Exact duplicate rows are also removed.

The original dataset is retained separately so that cancellation and return behavior can be analyzed independently.

In [19]:
df_clean=df.copy()
df_clean=df.drop_duplicates()

df_clean=df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
df_clean=df_clean[df_clean['Quantity'] > 0]
df_clean=df_clean[df_clean['UnitPrice'] > 0]
print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))

Original rows: 541909
Cleaned rows: 524878
Rows removed: 17031


In [21]:
df_clean['Revenue']=df_clean['UnitPrice'] * df_clean['Quantity']
df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [24]:
print("Cleaned dataset shape:", df_clean.shape)
print("Total Revenue: £{:,.2f}".format(df_clean['Revenue'].sum()))

Cleaned dataset shape: (524878, 9)
Total Revenue: £10,642,110.80


In [25]:
# Create time-based and business metrics
df_clean['Year']=df_clean['InvoiceDate'].dt.year
df_clean['Month']=df_clean['InvoiceDate'].dt.month
df_clean['Month_Name']=df_clean['InvoiceDate'].dt.month_name()
df_clean['Quarter']=df_clean['InvoiceDate'].dt.quarter
df_clean['Day']=df_clean['InvoiceDate'].dt.day
df_clean['Day_Name']=df_clean['InvoiceDate'].dt.day_name()

df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Year,Month,Month_Name,Quarter,Day,Day_Name
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,December,4,1,Wednesday
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,December,4,1,Wednesday
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,December,4,1,Wednesday
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,December,4,1,Wednesday
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,December,4,1,Wednesday


In [26]:
print("Start Date:", df_clean['InvoiceDate'].min())
print("End Date:", df_clean['InvoiceDate'].max())

Start Date: 2010-12-01 08:26:00
End Date: 2011-12-09 12:50:00


In [27]:
print('Unique Invoices:',df_clean['InvoiceNo'].nunique())
print('Unique Products:',df_clean['StockCode'].nunique())
print('Unique Customers:',df_clean['CustomerID'].nunique())
print('Unique Countries:',df_clean['Country'].nunique())


Unique Invoices: 19960
Unique Products: 3922
Unique Customers: 4338
Unique Countries: 38


In [28]:
df_clean.to_csv('online retail_cleaned.csv',index=False)
print('Cleaned dataset saved sucessfully')

Cleaned dataset saved sucessfully


In [31]:
import os

print('File exists:',os.path.exists('online retail_cleaned.csv'))

File exists: True
